In [1]:
!pip install scikit-learn pandas

import pandas as pd
import numpy as np

In [4]:
from google.colab import files
uploaded = files.upload()

Saving Fake.csv to Fake.csv
Saving True.csv to True.csv


In [5]:
fake = pd.read_csv('Fake.csv')
true = pd.read_csv('True.csv')

fake['label'] = 0   # Fake = 0
true['label'] = 1   # Real = 1

data = pd.concat([fake, true], axis=0)
data = data.sample(frac=1).reset_index(drop=True)

data.head()

,title,text,subject,date,label
0,HILLARY CLINTON CRASHING IN POLLS: Moves To Ob...,"So, the working people of America are basicall...",politics,"Aug 10, 2015",0
1,UNREAL! WATCH JOE BIDEN Point Out The Guy Who ...,Are you kidding me? THIS is the guy in charge ...,politics,"Aug 15, 2016",0
2,“MAXINE WATERS IN A GLITTERY COWBOY HAT” Goes ...,The left is going ballistic over supposed word...,left-news,"Oct 18, 2017",0
3,WOW! HILLARY’S GOT GOVERNOR OF IOWA Shakin’ In...,Wow!Hillary s got the governor of Iowa shaking...,politics,"Feb 1, 2016",0
4,State Department informed of court ruling on T...,WASHINGTON (Reuters) - The U.S. State Departme...,politicsNews,"February 4, 2017",1


In [6]:
data = data[['text', 'label']]

data.isnull().sum()

data = data.dropna()

In [7]:
from sklearn.model_selection import train_test_split

X = data['text']
y = data['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words='english', max_df=0.7)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_tfidf, y_train)

y_pred_lr = lr.predict(X_test_tfidf)

print("🔹 Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

🔹 Logistic Regression Accuracy: 0.9854120267260579
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      4621
           1       0.98      0.99      0.98      4359

    accuracy                           0.99      8980
   macro avg       0.99      0.99      0.99      8980
weighted avg       0.99      0.99      0.99      8980



In [10]:
from sklearn.svm import LinearSVC

svm = LinearSVC()
svm.fit(X_train_tfidf, y_train)

y_pred_svm = svm.predict(X_test_tfidf)

print("🔹 SVM Accuracy:", accuracy_score(y_test, y_pred_svm))
print(classification_report(y_test, y_pred_svm))

🔹 SVM Accuracy: 0.994097995545657
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      4621
           1       0.99      0.99      0.99      4359

    accuracy                           0.99      8980
   macro avg       0.99      0.99      0.99      8980
weighted avg       0.99      0.99      0.99      8980



In [11]:
def predict_news(text):
    text_tfidf = vectorizer.transform([text])
    pred_lr = lr.predict(text_tfidf)[0]
    pred_svm = svm.predict(text_tfidf)[0]

    print("Logistic Regression:", "Real" if pred_lr==1 else "Fake")
    print("SVM:", "Real" if pred_svm==1 else "Fake")

# Example
predict_news("Government announces new economic policy to boost growth")

Logistic Regression: Real
SVM: Fake
